# Week 4 – Text Embeddings
## Generating and Comparing Text Embeddings using Sentence Transformers

**Objective:** Generate sentence embeddings and compare sentence similarity using cosine similarity.

**Model:** `all-MiniLM-L6-v2`


## 1. Import the libraries

If a package is missing, run `python -m pip install -r requirements.txt` in the project terminal, then restart the notebook kernel.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


## 2. Load the dataset

In [2]:
BASE_DIR = Path.cwd()
# If the notebook is opened from the notebook folder, use its parent as the project folder.
if not (BASE_DIR / "dataset").exists() and (BASE_DIR.parent / "dataset").exists():
    BASE_DIR = BASE_DIR.parent

DATASET_DIR = BASE_DIR / "dataset"
df = pd.read_csv(DATASET_DIR / "generated_text.csv")
df.head()


,sentence_id,text
0,1,Artificial intelligence enables computers to p...
1,2,Machine learning allows computers to learn pat...
2,3,Deep learning is a branch of machine learning ...
3,4,Natural language processing helps computers un...
4,5,Text embeddings represent sentences as numeric...


## 3. Generate sentence embeddings

In [3]:
sentences = df["text"].dropna().astype(str).tolist()
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(
    sentences,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Number of sentences:", len(sentences))
print("Embedding shape:", embeddings.shape)
print("Embedding dimension:", embeddings.shape[1])
np.save(DATASET_DIR / "embeddings.npy", embeddings)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of sentences: 20
Embedding shape: (20, 384)
Embedding dimension: 384


## 4. Calculate cosine similarity

In [4]:
# Normalized embeddings allow cosine similarity to be calculated using a dot product.
similarity_matrix = embeddings @ embeddings.T
similarity_matrix_df = pd.DataFrame(
    similarity_matrix,
    index=df["sentence_id"],
    columns=df["sentence_id"]
)
similarity_matrix_df.head()


sentence_id,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
sentence_id,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.596430,0.418802,0.565806,0.222741,0.224482,0.099971,0.442082,0.429404,0.259896,0.220967,0.239320,0.081384,0.100695,0.143385,-0.046622,0.165772,0.143559,0.350957,0.364768
2,0.596430,1.000000,0.531747,0.534894,0.236411,0.267186,0.180018,0.412210,0.333464,0.530952,0.284267,0.378122,0.095409,0.071870,0.176322,-0.013294,0.107265,0.157307,0.307854,0.290642
3,0.418802,0.531747,1.000000,0.332573,0.245747,0.234580,0.055599,0.293096,0.490480,0.297257,0.139936,0.184475,0.141546,0.045877,0.088247,0.014660,0.048112,0.085617,0.241477,0.211004
4,0.565806,0.534894,0.332573,1.000000,0.388274,0.470873,0.151147,0.347948,0.202949,0.256390,0.265610,0.259324,0.090654,0.080319,0.091274,0.024713,0.074171,0.177020,0.216743,0.260104
5,0.222741,0.236411,0.245747,0.388274,1.000000,0.685898,0.299232,0.093511,0.274501,0.146376,0.124600,0.167515,0.089195,0.001555,0.030645,-0.046492,-0.012786,0.088928,0.074148,0.024961


## 5. Save all unique sentence-pair scores and display the top five

In [5]:
rows = []
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        rows.append({
            "sentence_id_1": int(df.iloc[i]["sentence_id"]),
            "sentence_id_2": int(df.iloc[j]["sentence_id"]),
            "sentence_1": sentences[i],
            "sentence_2": sentences[j],
            "cosine_similarity": float(similarity_matrix[i, j])
        })

scores = pd.DataFrame(rows).sort_values("cosine_similarity", ascending=False)
scores.to_csv(DATASET_DIR / "similarity_scores.csv", index=False)
scores.head(5)


,sentence_id_1,sentence_id_2,sentence_1,sentence_2,cosine_similarity
70,5,6,Text embeddings represent sentences as numeric...,Sentence transformers create meaningful vector...,0.685898
0,1,2,Artificial intelligence enables computers to p...,Machine learning allows computers to learn pat...,0.596430
2,1,4,Artificial intelligence enables computers to p...,Natural language processing helps computers un...,0.565806
20,2,4,Machine learning allows computers to learn pat...,Natural language processing helps computers un...,0.534894
19,2,3,Machine learning allows computers to learn pat...,Deep learning is a branch of machine learning ...,0.531747


## 6. Compare selected sentence pairs

In [6]:
pairs = pd.read_csv(DATASET_DIR / "sentence_pairs.csv")
id_to_position = {int(row["sentence_id"]): pos for pos, (_, row) in enumerate(df.iterrows())}

pair_results = []
for _, pair in pairs.iterrows():
    id1, id2 = int(pair["sentence_id_1"]), int(pair["sentence_id_2"])
    i, j = id_to_position[id1], id_to_position[id2]
    pair_results.append({
        "sentence_id_1": id1,
        "sentence_id_2": id2,
        "sentence_1": sentences[i],
        "sentence_2": sentences[j],
        "cosine_similarity": float(similarity_matrix[i, j])
    })

pair_results_df = pd.DataFrame(pair_results).sort_values(
    "cosine_similarity", ascending=False
)
pair_results_df.to_csv(DATASET_DIR / "sentence_pairs.csv", index=False)
pair_results_df


,sentence_id_1,sentence_id_2,sentence_1,sentence_2,cosine_similarity
3,5,6,Text embeddings represent sentences as numeric...,Sentence transformers create meaningful vector...,0.685898
0,1,2,Artificial intelligence enables computers to p...,Machine learning allows computers to learn pat...,0.596430
10,4,1,Natural language processing helps computers un...,Artificial intelligence enables computers to p...,0.565806
1,2,3,Machine learning allows computers to learn pat...,Deep learning is a branch of machine learning ...,0.531747
7,10,2,Data cleaning improves the quality of machine ...,Machine learning allows computers to learn pat...,0.530952
5,1,3,Artificial intelligence enables computers to p...,Deep learning is a branch of machine learning ...,0.418802
2,4,5,Natural language processing helps computers un...,Text embeddings represent sentences as numeric...,0.388274
8,13,14,Renewable energy sources include solar power a...,"Plants need sunlight, water, and nutrients to ...",0.332135
13,15,14,The Earth revolves around the Sun once every y...,"Plants need sunlight, water, and nutrients to ...",0.312153
14,20,8,Cloud computing provides computing services ov...,Python is widely used for data science and art...,0.308408


## 7. Observation and conclusion

Text embeddings represent sentence meaning as numerical vectors. The model used here produces 384-dimensional vectors. Cosine similarity helps compare the semantic relationship between sentence pairs; higher values generally indicate greater similarity. Embeddings are useful for semantic search and Retrieval-Augmented Generation (RAG).
